## **End-to-End RAG System for Nutrition Question Answering**

## Project Overview

In this assignment, we built a Retrieval-Augmented Generation (RAG) system for factual question answering in the domain of human nutrition. The system uses an open-source nutrition textbook as its knowledge source. Given a question, it retrieves relevant text chunks from the document and uses a language model to generate a concise answer based on the retrieved context.

The pipeline is organized into two main stages: document preprocessing and embedding creation, followed by retrieval and answer generation.

## Environment Setup

This notebook was developed in Google Colab with GPU support. The required libraries were installed for PDF processing, text chunking, embedding generation, retrieval, and language model inference.

In [1]:
!pip uninstall -y torch torchvision torchaudio
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128

Found existing installation: torch 2.11.0+cu128
Uninstalling torch-2.11.0+cu128:
  Successfully uninstalled torch-2.11.0+cu128
Found existing installation: torchvision 0.26.0+cu128
Uninstalling torchvision-0.26.0+cu128:
  Successfully uninstalled torchvision-0.26.0+cu128
Found existing installation: torchaudio 2.11.0+cu128
Uninstalling torchaudio-2.11.0+cu128:
  Successfully uninstalled torchaudio-2.11.0+cu128
Looking in indexes: https://download.pytorch.org/whl/cu128
  Using cached https://download-r2.pytorch.org/whl/cu128/torch-2.11.0%2Bcu128-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (29 kB)
  Using cached https://download-r2.pytorch.org/whl/cu128/torchvision-0.26.0%2Bcu128-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (5.5 kB)
  Using cached https://download-r2.pytorch.org/whl/cu128/torchaudio-2.11.0%2Bcu128-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (6.9 kB)
Using cached https://download-r2.pytorch.org/whl/cu128/torch-2.11.0%2Bcu128-cp312-cp312-manylinux_2_28_x86_64.whl (

In [2]:
# Perform Google Colab installs (if running in Google Colab)
import os

if "COLAB_GPU" in os.environ:
    print("[INFO] Running in Google Colab, installing requirements.")
    !pip install PyMuPDF # for reading PDFs with Python
    !pip install tqdm # for progress bars
    !pip install sentence-transformers # for embedding models
    !pip install accelerate # for quantization model loading
    !pip install bitsandbytes # for quantizing models (less storage space)
    !pip install flash-attn --no-build-isolation # for faster attention mechanism = faster LLM inference

[INFO] Running in Google Colab, installing requirements.
  Using cached flash_attn-2.8.3.tar.gz (8.4 MB)
  Preparing metadata (setup.py) ... done
  error: subprocess-exited-with-error
  
  × python setup.py bdist_wheel did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for flash-attn
  Running setup.py clean for flash-attn
Failed to build flash-attn
ERROR: ERROR: Failed to build installable wheels for some pyproject.toml based projects (flash-attn)


## 1. Document Processing and Embedding Creation

The first stage of the pipeline prepares the source document for retrieval. This includes loading the PDF, extracting text, splitting the text into smaller chunks, and converting those chunks into embeddings for later similarity search.

### Data Ingestion: PDF Loading

In this step, the source document is loaded into the pipeline. The dataset used in this project is the open-source textbook *Human Nutrition: 2020 Edition*, which is available online as a downloadable PDF.

If the document is not already available locally, it is programmatically downloaded from the source URL. The PDF is then opened using the PyMuPDF (fitz) library, which allows efficient access to the document content.

Each page of the PDF is processed to extract textual data, which is stored for further preprocessing and analysis in subsequent steps.

In [3]:
# Download PDF file
import os
import requests

# Create folder for raw data
os.makedirs("data/raw", exist_ok=True)

# Get PDF document
pdf_path = "data/raw/human-nutrition-text.pdf"

# Download PDF if it doesn't already exist
if not os.path.exists(pdf_path):
    print("File doesn't exist, downloading...")

    # The URL of the PDF
    url = "https://pressbooks.oer.hawaii.edu/humannutrition2/open/download?type=pdf"

    # Send a GET request
    response = requests.get(url)

    # Check if successful
    if response.status_code == 200:
        with open(pdf_path, "wb") as file:
            file.write(response.content)
        print(f"The file has been downloaded and saved as {pdf_path}")
    else:
        print(f"Failed to download the file. Status code: {response.status_code}")
else:
    print(f"File {pdf_path} exists.")

File data/raw/human-nutrition-text.pdf exists.


In [4]:
# Requires !pip install PyMuPDF
import fitz
from tqdm.auto import tqdm

def text_formatter(text: str) -> str:
    """Performs minor formatting on text."""
    cleaned_text = text.replace("\n", " ").strip()
    return cleaned_text

def open_and_read_pdf(pdf_path: str) -> list[dict]:
    """
    Opens a PDF file, reads its text content page by page, and collects statistics.
    """
    doc = fitz.open(pdf_path)
    pages_and_texts = []

    for page_number, page in tqdm(enumerate(doc)):
        text = page.get_text()
        text = text_formatter(text)

        pages_and_texts.append({
            "page_number": page_number,  # FIXED
            "page_char_count": len(text),
            "page_word_count": len(text.split(" ")),
            "page_sentence_count_raw": len(text.split(". ")),
            "page_token_count": len(text) / 4,
            "text": text
        })

    return pages_and_texts

pages_and_texts = open_and_read_pdf(pdf_path=pdf_path)
pages_and_texts[:2]

0it [00:00, ?it/s]

[{'page_number': 0,
  'page_char_count': 29,
  'page_word_count': 4,
  'page_sentence_count_raw': 1,
  'page_token_count': 7.25,
  'text': 'Human Nutrition: 2020 Edition'},
 {'page_number': 1,
  'page_char_count': 0,
  'page_word_count': 1,
  'page_sentence_count_raw': 1,
  'page_token_count': 0.0,
  'text': ''}]

In [5]:
import random

random.sample(pages_and_texts, k=3)

[{'page_number': 1005,
  'page_char_count': 1092,
  'page_word_count': 166,
  'page_sentence_count_raw': 13,
  'page_token_count': 273.0,
  'text': 'for vegetarian athletes. (See Table 16.2 “The Recommended Protein  Intakes for Individuals” )  Micronutrient Needs  Vitamins and minerals are essential for energy metabolism, the  delivery of oxygen, protection against oxidative damage, and the  repair of body structures. When exercise increases, the amount  of many vitamins and minerals needed are also increased due to  the excess loss in nutrients. \xa0Currently, there is not special  micronutrient recommendations made for athletes but most  athletes will meet their needs by consuming a balanced diet that  meets their energy needs. Because the energy needs of athletes  increase, they often consume extra vitamins and minerals. The  major micronutrients of concern for athletes include iron, calcium,  vitamin D, and some antioxidants. 6  6.\xa0Nutrition and Athletic Performance. (2016). Ame

### Text Statistics and Analysis

After extracting the text from the document, a basic analysis is performed to understand its size and structure. This includes computing statistics such as character count, word count, and an approximate token count for each page.

These statistics provide insight into the distribution and length of the extracted text, which is important for determining an appropriate chunking strategy.

Since embedding models have limits on the amount of text they can process at once (e.g., all-mpnet-base-v2 supports inputs up to 384 tokens), this analysis helps ensure that the text is later divided into suitable chunk sizes without losing important information.

In [6]:
import pandas as pd

df = pd.DataFrame(pages_and_texts)
df.head()

,page_number,page_char_count,page_word_count,page_sentence_count_raw,page_token_count,text
0,0,29,4,1,7.25,Human Nutrition: 2020 Edition
1,1,0,1,1,0.00,
2,2,320,54,1,80.00,Human Nutrition: 2020 Edition UNIVERSITY OF ...
3,3,212,32,1,53.00,Human Nutrition: 2020 Edition by University of...
4,4,797,145,2,199.25,Contents Preface University of Hawai‘i at Mā...


In [7]:
# Get stats
df.describe().round(2)

,page_number,page_char_count,page_word_count,page_sentence_count_raw,page_token_count
count,1208.00,1208.00,1208.00,1208.00,1208.00
mean,603.50,1148.00,198.30,9.97,287.00
std,348.86,560.38,95.76,6.19,140.10
min,0.00,0.00,1.00,1.00,0.00
25%,301.75,762.00,134.00,4.00,190.50
50%,603.50,1231.50,214.50,10.00,307.88
75%,905.25,1603.50,271.00,14.00,400.88
max,1207.00,2308.00,429.00,32.00,577.00


### Sentence Splitting

To prepare the text for embedding, the extracted content is first divided into smaller units. Instead of working with entire pages, the text is split into individual sentences using the spaCy NLP library.

Sentence-level splitting provides finer granularity, making it easier to group related information and improve retrieval accuracy. It also ensures that large blocks of text are broken down into manageable segments for further processing.

These sentences are later grouped into fixed-size chunks, which are used as the input for embedding and retrieval in the RAG pipeline.

In [8]:
from spacy.lang.en import English # see https://spacy.io/usage for install instructions

nlp = English()

# Add a sentencizer pipeline, see https://spacy.io/api/sentencizer/
nlp.add_pipe("sentencizer")

# Create a document instance as an example
doc = nlp("This is a sentence. This another sentence.")
assert len(list(doc.sents)) == 2

# Access the sentences of the document
list(doc.sents)

[This is a sentence., This another sentence.]

In [9]:
for item in tqdm(pages_and_texts):
    item["sentences"] = list(nlp(item["text"]).sents)

    # Make sure all sentences are strings
    item["sentences"] = [str(sentence) for sentence in item["sentences"]]

    # Count the sentences
    item["page_sentence_count_spacy"] = len(item["sentences"])

  0%|          | 0/1208 [00:00<?, ?it/s]

In [10]:
# Inspect an example
random.sample(pages_and_texts, k=1)

[{'page_number': 70,
  'page_char_count': 201,
  'page_word_count': 33,
  'page_sentence_count_raw': 2,
  'page_token_count': 50.25,
  'text': 'An interactive or media element has been  excluded from this version of the text. You can  view it online here:  http://pressbooks.oer.hawaii.edu/ humannutrition2/?p=55  Lifestyles and Nutrition  |  29',
  'sentences': ['An interactive or media element has been  excluded from this version of the text.',
   'You can  view it online here:  http://pressbooks.oer.hawaii.edu/ humannutrition2/?p=55  Lifestyles and Nutrition  |  29'],
  'page_sentence_count_spacy': 2}]

In [11]:
df = pd.DataFrame(pages_and_texts)
df.describe().round(2)

,page_number,page_char_count,page_word_count,page_sentence_count_raw,page_token_count,page_sentence_count_spacy
count,1208.00,1208.00,1208.00,1208.00,1208.00,1208.00
mean,603.50,1148.00,198.30,9.97,287.00,10.32
std,348.86,560.38,95.76,6.19,140.10,6.30
min,0.00,0.00,1.00,1.00,0.00,0.00
25%,301.75,762.00,134.00,4.00,190.50,5.00
50%,603.50,1231.50,214.50,10.00,307.88,10.00
75%,905.25,1603.50,271.00,14.00,400.88,15.00
max,1207.00,2308.00,429.00,32.00,577.00,28.00


### Text Chunking

After splitting the text into sentences, the next step is to group these sentences into fixed-size chunks. This process, known as chunking, helps organize the text into manageable units for embedding and retrieval.

Chunking ensures that each segment remains within the token limits of the embedding model, while still preserving enough context for meaningful semantic representation. In this pipeline, a fixed-size approach is used, where a set number of sentences are combined to form each chunk.

This results in uniformly sized text segments that are well-suited for efficient similarity search and downstream question-answering tasks.

In [12]:
# Define split size to turn groups of sentences into chunks
# Smaller chunks usually improve factual retrieval for QA
num_sentence_chunk_size = 6

# Create a function that recursively splits a list into desired sizes
def split_list(input_list: list,
               slice_size: int) -> list[list[str]]:
    """
    Splits the input_list into sublists of size slice_size.

    Example:
    A list of 17 sentences with slice_size=6 becomes:
    [[0:6], [6:12], [12:17]]
    """
    return [input_list[i:i + slice_size] for i in range(0, len(input_list), slice_size)]

# Loop through pages and texts and split sentences into chunks
for item in tqdm(pages_and_texts):
    item["sentence_chunks"] = split_list(
        input_list=item["sentences"],
        slice_size=num_sentence_chunk_size
    )
    item["num_chunks"] = len(item["sentence_chunks"])

  0%|          | 0/1208 [00:00<?, ?it/s]

In [13]:
# Sample an example from the group (note: many samples have only 1 chunk as they have <=10 sentences total)
random.sample(pages_and_texts, k=1)

[{'page_number': 344,
  'page_char_count': 804,
  'page_word_count': 148,
  'page_sentence_count_raw': 7,
  'page_token_count': 201.0,
  'text': 'Tools for Change  There are many  sources of omega-3 foods.  It is important to strike a proper balance between  omega-3 and omega-6 fats in your diet. Research suggests  that a diet that is too high in omega-6 fats distorts the  balance of proinflammatory agents, promoting chronic  inflammation and causing the potential for health problems  such as asthma, arthritis, allergies, or diabetes. Omega-6  fats compete with omega-3 fats for enzymes and will  actually replace omega-3 fats. The typical western diet is  characterized by an excessive consumption of foods high in  omega-6 fatty acids. To gain proper balance between the  two, increase your omega-3 fat intake by eating more fatty  fish or other sources of omega-3 fatty acids at least two  times per week.  The Role of Lipids in Food  |  303',
  'sentences': ['Tools for Change  There are ma

In [14]:
# Create a DataFrame to get stats
df = pd.DataFrame(pages_and_texts)
df.describe().round(2)

,page_number,page_char_count,page_word_count,page_sentence_count_raw,page_token_count,page_sentence_count_spacy,num_chunks
count,1208.00,1208.00,1208.00,1208.00,1208.00,1208.00,1208.00
mean,603.50,1148.00,198.30,9.97,287.00,10.32,2.15
std,348.86,560.38,95.76,6.19,140.10,6.30,1.07
min,0.00,0.00,1.00,1.00,0.00,0.00,0.00
25%,301.75,762.00,134.00,4.00,190.50,5.00,1.00
50%,603.50,1231.50,214.50,10.00,307.88,10.00,2.00
75%,905.25,1603.50,271.00,14.00,400.88,15.00,3.00
max,1207.00,2308.00,429.00,32.00,577.00,28.00,5.00


### Creating Chunk-Level Records

After chunking the text, each chunk is stored as an individual record along with its associated metadata. This includes information such as the page number and basic statistics for each chunk.

Organizing the data at the chunk level makes it easier to generate embeddings, perform retrieval, and trace each retrieved result back to its original location in the source document.

In [15]:
import re

# Split each chunk into its own item
pages_and_chunks = []

for item in tqdm(pages_and_texts):
    for sentence_chunk in item["sentence_chunks"]:
        chunk_dict = {}
        chunk_dict["page_number"] = item["page_number"]
        chunk_dict["chunk_id"] = f"chunk_{len(pages_and_chunks)}"

        # Join sentences into one chunk
        joined_sentence_chunk = " ".join(sentence_chunk).replace("  ", " ").strip()
        joined_sentence_chunk = re.sub(r"\s+", " ", joined_sentence_chunk)
        joined_sentence_chunk = re.sub(r"\.([A-Z])", r". \1", joined_sentence_chunk)

        chunk_dict["sentence_chunk"] = joined_sentence_chunk

        # Stats
        chunk_dict["chunk_char_count"] = len(joined_sentence_chunk)
        chunk_dict["chunk_word_count"] = len(joined_sentence_chunk.split())
        chunk_dict["chunk_token_count"] = len(joined_sentence_chunk) / 4  # rough estimate

        pages_and_chunks.append(chunk_dict)

# How many chunks do we have?
len(pages_and_chunks)

  0%|          | 0/1208 [00:00<?, ?it/s]

2603

In [16]:
# View a random sample
random.sample(pages_and_chunks, k=1)

[{'page_number': 199,
  'chunk_id': 'chunk_347',
  'sentence_chunk': 'reactions as it can store a large amount of heat, is electrically neutral, and has a pH of 7.0, meaning it is not acidic or basic. Additionally, water is involved in many enzymatic reactions as an agent to break bonds or, by its removal from a molecule, to form bonds. Water As a Lubricant/Shock Absorber Many may view the slimy products of a sneeze as gross, but sneezing is essential for removing irritants and could not take place without water. Mucus, which is not only essential to discharge nasal irritants, is also required for breathing, transportation of nutrients along the gastrointestinal tract, and elimination of waste materials through the rectum. Mucus is composed of more than 90 percent water and a front-line defense against injury and foreign invaders. It protects tissues from irritants, entraps pathogens, and contains immune-system cells that destroy pathogens.',
  'chunk_char_count': 884,
  'chunk_word_co

In [17]:
# Get stats about our chunks
df = pd.DataFrame(pages_and_chunks)
df.describe().round(2)

,page_number,chunk_char_count,chunk_word_count,chunk_token_count
count,2603.00,2603.00,2603.00,2603.00
mean,636.23,519.57,79.92,129.89
std,346.60,308.48,49.06,77.12
min,0.00,10.00,3.00,2.50
25%,334.50,243.00,34.00,60.75
50%,642.00,535.00,84.00,133.75
75%,944.50,742.00,115.00,185.50
max,1207.00,1647.00,280.00,411.75


In [18]:
# Show random chunks with under the minimum token threshold
# Lower threshold because short factual chunks can contain gold answers
min_token_length = 10

short_chunks = df[df["chunk_token_count"] <= min_token_length]

if len(short_chunks) > 0:
    sample_n = min(5, len(short_chunks))
    for row in short_chunks.sample(sample_n).iterrows():
        print(f'Chunk token count: {row[1]["chunk_token_count"]:.1f} | Text: {row[1]["sentence_chunk"]}')
else:
    print("No chunks under the minimum token threshold.")

Chunk token count: 5.2 | Text: Young Adulthood | 911
Chunk token count: 3.0 | Text: 176 | Sodium
Chunk token count: 4.0 | Text: Middle Age | 917
Chunk token count: 4.0 | Text: 916 | Middle Age
Chunk token count: 2.5 | Text: 656 | Iron


In [19]:
pages_and_chunks_over_min_token_len = df[
    df["chunk_token_count"] > min_token_length
].to_dict(orient="records")

pages_and_chunks_over_min_token_len[:2]

[{'page_number': 2,
  'chunk_id': 'chunk_1',
  'sentence_chunk': 'Human Nutrition: 2020 Edition UNIVERSITY OF HAWAI‘I AT MĀNOA FOOD SCIENCE AND HUMAN NUTRITION PROGRAM ALAN TITCHENAL, SKYLAR HARA, NOEMI ARCEO CAACBAY, WILLIAM MEINKE-LAU, YA-YUN YANG, MARIE KAINOA FIALKOWSKI REVILLA, JENNIFER DRAPER, GEMADY LANGFELDER, CHERYL GIBBY, CHYNA NICOLE CHUN, AND ALLISON CALABRESE',
  'chunk_char_count': 308,
  'chunk_word_count': 42,
  'chunk_token_count': 77.0},
 {'page_number': 3,
  'chunk_id': 'chunk_2',
  'sentence_chunk': 'Human Nutrition: 2020 Edition by University of Hawai‘i at Mānoa Food Science and Human Nutrition Program is licensed under a Creative Commons Attribution 4.0 International License, except where otherwise noted.',
  'chunk_char_count': 210,
  'chunk_word_count': 30,
  'chunk_token_count': 52.5}]

### Generating Embeddings

In this step, each text chunk is converted into a numerical representation, known as an embedding. Embeddings capture the semantic meaning of text, allowing similar pieces of text to have similar vector representations.

The sentence-transformers/all-mpnet-base-v2 model is used to generate embeddings for all text chunks. This model produces dense vector representations that can be efficiently compared using similarity measures.

Once generated, these embeddings are stored and used during retrieval to identify the most relevant chunks for a given query based on semantic similarity.

In [20]:
!pip uninstall -y torchcodec

In [21]:
import torch
from sentence_transformers import SentenceTransformer

device = "cuda" if torch.cuda.is_available() else "cpu"

embedding_model = SentenceTransformer(
    model_name_or_path="all-mpnet-base-v2",
    device=device
)

print("Embedding model loaded on:", device)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding model loaded on: cuda


In [22]:
%%time

texts = [item["sentence_chunk"] for item in pages_and_chunks_over_min_token_len]

embeddings = embedding_model.encode(
    texts,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True
)

for i, item in enumerate(pages_and_chunks_over_min_token_len):
    item["embedding"] = embeddings[i]

Batches:   0%|          | 0/80 [00:00<?, ?it/s]

CPU times: user 18.9 s, sys: 318 ms, total: 19.2 s
Wall time: 29.1 s


### Embedding Representation

Each text chunk is converted into a fixed-size embedding vector using the sentence-transformers/all-mpnet-base-v2 model. Each embedding has a dimensionality of 768, representing the semantic meaning of the text in a high-dimensional space.

These embeddings allow efficient comparison between queries and document chunks, as semantically similar texts are mapped closer together in the vector space.

The embeddings are computed once and stored along with their corresponding text chunks for efficient retrieval during the question-answering stage.

In [23]:
%%time

import torch

# Move model to GPU if available
device = "cuda" if torch.cuda.is_available() else "cpu"
embedding_model.to(device)

# Get all chunk texts
texts = [item["sentence_chunk"] for item in pages_and_chunks_over_min_token_len]

# Batch encode all chunks
embeddings = embedding_model.encode(
    texts,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True
)

# Store embeddings back into each chunk item
for i, item in enumerate(pages_and_chunks_over_min_token_len):
    item["embedding"] = embeddings[i]

Batches:   0%|          | 0/80 [00:00<?, ?it/s]

CPU times: user 23.5 s, sys: 96.5 ms, total: 23.6 s
Wall time: 29.1 s


In [24]:
# Turn text chunks into a single list
text_chunks = [item["sentence_chunk"] for item in pages_and_chunks_over_min_token_len]

In [25]:
# Save embeddings to file
import os

os.makedirs("artifacts", exist_ok=True)

text_chunks_and_embeddings_df = pd.DataFrame(pages_and_chunks_over_min_token_len)
embeddings_df_save_path = "artifacts/text_chunks_and_embeddings_df.csv"
text_chunks_and_embeddings_df.to_csv(embeddings_df_save_path, index=False)

In [26]:
# Import saved file and view
text_chunks_and_embedding_df_load = pd.read_csv(embeddings_df_save_path)
text_chunks_and_embedding_df_load.head()

,page_number,chunk_id,sentence_chunk,chunk_char_count,chunk_word_count,chunk_token_count,embedding
0,2,chunk_1,Human Nutrition: 2020 Edition UNIVERSITY OF HA...,308,42,77.00,[ 6.74242824e-02 9.02282000e-02 -5.09549864e-...
1,3,chunk_2,Human Nutrition: 2020 Edition by University of...,210,30,52.50,[ 5.52156083e-02 5.92139438e-02 -1.66167226e-...
2,4,chunk_3,Contents Preface University of Hawai‘i at Māno...,766,116,191.50,[ 2.79801544e-02 3.39814015e-02 -2.06426457e-...
3,5,chunk_4,Lifestyles and Nutrition University of Hawai‘i...,941,144,235.25,[ 6.82566911e-02 3.81275043e-02 -8.46855156e-...
4,6,chunk_5,The Cardiovascular System University of Hawai‘...,998,152,249.50,[ 3.30264196e-02 -8.49765539e-03 9.57158115e-...


### Embedding Design Considerations

The sentence-transformers/all-mpnet-base-v2 model is used to generate embeddings for both document chunks and user queries. This ensures that both are represented in the same vector space, allowing meaningful similarity comparisons during retrieval.

The model produces fixed-size embeddings of 768 dimensions and supports input sequences up to a limited token length. Therefore, the earlier chunking strategy ensures that all text segments remain within this limit while preserving sufficient contextual information.

Since the dataset size is relatively small, the generated embeddings are stored locally and reused during inference. This avoids recomputation and enables efficient similarity search without requiring a dedicated vector database.

## 2. RAG: Retrieval and Answer Generation

Retrieval-Augmented Generation (RAG) is a framework that improves question answering by combining information retrieval with language model generation.

The process consists of three main components. First, the **retrieval** step identifies the most relevant text chunks from the document based on the input query. Next, the **augmentation** step incorporates these retrieved chunks into the prompt, providing the model with context. Finally, the **generation** step uses a language model to produce an answer grounded in the retrieved information.

This approach helps improve factual accuracy by reducing reliance on the model’s internal knowledge and instead grounding responses in external, domain-specific data.

In [27]:
import random

import torch
import numpy as np
import pandas as pd

device = "cuda" if torch.cuda.is_available() else "cpu"

# Load embeddings file
text_chunks_and_embedding_df = pd.read_csv("artifacts/text_chunks_and_embeddings_df.csv")

# Convert embedding column back to numpy array
text_chunks_and_embedding_df["embedding"] = text_chunks_and_embedding_df["embedding"].apply(
    lambda x: np.fromstring(x.strip("[]"), sep=" ")
)

# Convert to list of dicts (IMPORTANT: keep correct variable name)
pages_and_chunks_over_min_token_len = text_chunks_and_embedding_df.to_dict(orient="records")

# Convert embeddings to torch tensor
embeddings = torch.tensor(
    np.array(text_chunks_and_embedding_df["embedding"].tolist()),
    dtype=torch.float32
).to(device)

embeddings.shape

torch.Size([2537, 768])

In [28]:
text_chunks_and_embedding_df.head()

,page_number,chunk_id,sentence_chunk,chunk_char_count,chunk_word_count,chunk_token_count,embedding
0,2,chunk_1,Human Nutrition: 2020 Edition UNIVERSITY OF HA...,308,42,77.00,"[0.0674242824, 0.0902282, -0.00509549864, -0.0..."
1,3,chunk_2,Human Nutrition: 2020 Edition by University of...,210,30,52.50,"[0.0552156083, 0.0592139438, -0.0166167226, -0..."
2,4,chunk_3,Contents Preface University of Hawai‘i at Māno...,766,116,191.50,"[0.0279801544, 0.0339814015, -0.0206426457, 0...."
3,5,chunk_4,Lifestyles and Nutrition University of Hawai‘i...,941,144,235.25,"[0.0682566911, 0.0381275043, -0.00846855156, -..."
4,6,chunk_5,The Cardiovascular System University of Hawai‘...,998,152,249.50,"[0.0330264196, -0.00849765539, 0.00957158115, ..."


In [29]:
embeddings[0]

tensor([ 6.7424e-02,  9.0228e-02, -5.0955e-03, -3.1755e-02,  7.3908e-02,
         3.5198e-02, -1.9799e-02,  4.6769e-02,  5.3573e-02,  5.0123e-03,
         3.3393e-02, -1.6222e-03,  1.7608e-02,  3.6265e-02, -3.1665e-04,
        -1.0712e-02,  1.5426e-02,  2.6218e-02,  2.7766e-03,  3.6494e-02,
        -4.4411e-02,  1.8936e-02,  4.9012e-02,  1.6402e-02, -4.8578e-02,
         3.1829e-03,  2.7299e-02, -2.0476e-03, -1.2283e-02, -7.2805e-02,
         1.2045e-02,  1.0730e-02,  2.1000e-03, -8.1777e-02,  2.6783e-06,
        -1.8143e-02, -1.2080e-02,  2.4717e-02, -6.2747e-02,  7.3544e-02,
         2.2162e-02, -3.2877e-02, -1.8010e-02,  2.2295e-02,  5.6137e-02,
         1.7951e-03,  5.2593e-02, -3.3174e-03, -8.3387e-03, -1.0628e-02,
         2.3192e-03, -2.2393e-02, -1.5301e-02, -9.9306e-03,  4.6532e-02,
         3.5747e-02, -2.5476e-02,  2.6369e-02,  3.7491e-03, -3.8268e-02,
         2.5833e-02,  4.1287e-02,  2.5818e-02,  3.3297e-02, -2.5178e-02,
         4.5152e-02,  4.4905e-04, -9.9662e-02,  4.9

In [30]:
from sentence_transformers import util, SentenceTransformer

embedding_model = SentenceTransformer(model_name_or_path="all-mpnet-base-v2",
                                      device=device) # choose the device to load the model to

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [31]:
# 1. Define the query
# Note: This could be anything. But since we're working with a nutrition textbook, we'll stick with nutrition-based queries.
query = "What are the functions of macronutrients?"
print(f"Query: {query}")

# 2. Embed the query to the same numerical space as the text examples
# Note: It's important to embed your query with the same model you embedded your examples with.
query_embedding = embedding_model.encode(query, convert_to_tensor=True)

# 3. Get similarity scores with the dot product (we'll time this for fun)
from time import perf_counter as timer

start_time = timer()
dot_scores = util.dot_score(a=query_embedding, b=embeddings)[0]
end_time = timer()

print(f"Time take to get scores on {len(embeddings)} embeddings: {end_time-start_time:.5f} seconds.")

# 4. Get the top-k results (we'll keep this to 5)
top_results_dot_product = torch.topk(dot_scores, k=5)
top_results_dot_product

Query: What are the functions of macronutrients?
Time take to get scores on 2537 embeddings: 0.02201 seconds.


torch.return_types.topk(
values=tensor([0.7303, 0.7181, 0.6685, 0.6568, 0.6550], device='cuda:0'),
indices=tensor([45, 54, 43, 59, 58], device='cuda:0'))

In [32]:
# Define helper function to print wrapped text
import textwrap

def print_wrapped(text, wrap_length=80):
    wrapped_text = textwrap.fill(text, wrap_length)
    print(wrapped_text)

In [33]:
print(f"Query: '{query}'\n")
print("Results:")

for score, idx in zip(top_results_dot_product[0], top_results_dot_product[1]):
    print(f"Score: {score:.4f}")
    print("Text:")
    print_wrapped(pages_and_chunks_over_min_token_len[idx]["sentence_chunk"])
    print(f"Page number: {pages_and_chunks_over_min_token_len[idx]['page_number']}")
    print("\n")

Query: 'What are the functions of macronutrients?'

Results:
Score: 0.7303
Text:
Macronutrients Nutrients that are needed in large amounts are called
macronutrients. There are three classes of macronutrients: carbohydrates,
lipids, and proteins. These can be metabolically processed into cellular energy.
The energy from macronutrients comes from their chemical bonds. This chemical
energy is converted into cellular energy that is then utilized to perform work,
allowing our bodies to conduct their basic functions. A unit of measurement of
food energy is the calorie.
Page number: 46


Score: 0.7181
Text:
Micronutrients Micronutrients are nutrients required by the body in lesser
amounts, but are still essential for carrying out bodily functions.
Micronutrients include all the essential minerals and vitamins. There are
sixteen essential minerals and thirteen vitamins (See Table 1.1 “Minerals and
Their Major Functions” and Table 1.2 “Vitamins and Their Major Functions” for a
complete list and

### Similarity Measurement

To retrieve relevant text chunks, similarity between the query embedding and document embeddings is computed. This is done using vector similarity measures, which quantify how close two embeddings are in the semantic space.

In this pipeline, similarity is computed using the dot product, which is efficient and effective for comparing embeddings generated by the all-mpnet-base-v2 model. Since the embeddings produced by this model are normalized, dot product and cosine similarity yield equivalent results.

These similarity scores are used to rank all text chunks, and the top-k most relevant chunks are selected for the next stage of the RAG pipeline.

In [34]:
def retrieve_relevant_resources(query: str,
                                embeddings: torch.tensor,
                                model: SentenceTransformer = embedding_model,
                                n_resources_to_return: int = 5,
                                print_time: bool = True):

    query_embedding = model.encode(query, convert_to_tensor=True)

    start_time = timer()
    dot_scores = util.dot_score(query_embedding, embeddings)[0]
    end_time = timer()

    if print_time:
        print(f"[INFO] Time taken to get scores on {len(embeddings)} embeddings: {end_time-start_time:.5f} seconds.")

    scores, indices = torch.topk(dot_scores, k=n_resources_to_return)
    return scores, indices


def tokenize_for_overlap(text):
    text = text.lower()
    text = re.sub(r"[^\w\s]", " ", text)
    return text.split()


def lexical_overlap_score(query, chunk):
    q_tokens = set(tokenize_for_overlap(query))
    c_tokens = set(tokenize_for_overlap(chunk))
    return len(q_tokens & c_tokens)


def retrieve_relevant_resources_reranked(query: str,
                                         embeddings: torch.tensor,
                                         model: SentenceTransformer = embedding_model,
                                         initial_k: int = 10,
                                         final_k: int = 5,
                                         print_time: bool = True):

    query_embedding = model.encode(query, convert_to_tensor=True)

    start_time = timer()
    dot_scores = util.dot_score(query_embedding, embeddings)[0]
    end_time = timer()

    if print_time:
        print(f"[INFO] Time taken to get scores on {len(embeddings)} embeddings: {end_time-start_time:.5f} seconds.")

    top_scores, top_indices = torch.topk(dot_scores, k=initial_k)

    candidates = []
    for score, idx in zip(top_scores, top_indices):
        idx = int(idx)
        chunk = pages_and_chunks_over_min_token_len[idx]["sentence_chunk"]
        overlap = lexical_overlap_score(query, chunk)

        # Dense score + light lexical rerank
        combined_score = float(score.cpu()) + 0.15 * overlap
        candidates.append((combined_score, float(score.cpu()), idx))

    candidates = sorted(candidates, key=lambda x: x[0], reverse=True)[:final_k]

    final_scores = torch.tensor([x[1] for x in candidates])
    final_indices = torch.tensor([x[2] for x in candidates])

    return final_scores, final_indices


def print_top_results_and_scores(query: str,
                                 embeddings: torch.tensor,
                                 pages_and_chunks: list[dict] = pages_and_chunks_over_min_token_len,
                                 n_resources_to_return: int = 5):

    scores, indices = retrieve_relevant_resources_reranked(
        query=query,
        embeddings=embeddings,
        initial_k=10,
        final_k=n_resources_to_return
    )

    print(f"Query: {query}\n")
    print("Results:")

    for score, index in zip(scores, indices):
        print(f"Score: {float(score):.4f}")
        print_wrapped(pages_and_chunks[int(index)]["sentence_chunk"])
        print(f"Page number: {pages_and_chunks[int(index)]['page_number']}")
        print("\n")

In [35]:
query = "symptoms of pellagra"

# Get just the scores and indices of top related results
scores, indices = retrieve_relevant_resources(query=query,
                                              embeddings=embeddings)
scores, indices

[INFO] Time taken to get scores on 2537 embeddings: 0.00011 seconds.


(tensor([0.3194, 0.3025, 0.2986, 0.2891, 0.2884], device='cuda:0'),
 tensor([1212, 1851, 1260, 1936, 2128], device='cuda:0'))

In [36]:
# Print out the texts of the top scores
print_top_results_and_scores(query=query,
                             embeddings=embeddings)

[INFO] Time taken to get scores on 2537 embeddings: 0.00009 seconds.
Query: symptoms of pellagra

Results:
Score: 0.2986
In the United States, B-vitamin deficiencies are rare; however in the nineteenth
century some vitamin-B deficiencies plagued many people in North America. Niacin
deficiency, also known as pellagra, was prominent in poorer Americans whose main
dietary staple was refined cornmeal. Its symptoms were severe and included
diarrhea, dermatitis, dementia, and even death. Some of the health consequences
of pellagra are the result of niacin being in insufficient supply to support the
body’s metabolic functions. Learning Activities Technology Note: The second
edition of the Human Nutrition Open Educational Resource (OER) textbook features
interactive learning activities. These activities are available in the web-based
textbook and not available in the downloadable versions (EPUB, Digital PDF,
Print_PDF, or Open Document).
Page number: 632


Score: 0.2881
NADPH is also required 

### Language Model for Answer Generation

To generate answers, a Large Language Model (LLM) is used as the final component of the RAG pipeline. The role of the LLM is to produce a response based on the input query and the retrieved context.

In this system, the prompt consists of the user query combined with the most relevant text chunks retrieved from the document. This ensures that the generated answer is grounded in domain-specific information rather than relying solely on the model’s internal knowledge.

The pipeline uses the instruction-tuned **Google Gemma 7B model (gemma-7b-it)** for generation, which is executed locally on the available hardware. This allows efficient inference while maintaining control over the data and execution process.

In [37]:
import torch

if torch.cuda.is_available():
    gpu_memory_bytes = torch.cuda.get_device_properties(0).total_memory
    gpu_memory_gb = round(gpu_memory_bytes / (2**30))
    print(f"Available GPU memory: {gpu_memory_gb} GB")
else:
    print("No GPU available.")

Available GPU memory: 15 GB


In [38]:
# Choose model based on available hardware

if torch.cuda.is_available():
    gpu_memory_bytes = torch.cuda.get_device_properties(0).total_memory
    gpu_memory_gb = round(gpu_memory_bytes / (2**30))
    print(f"GPU memory: {gpu_memory_gb} GB")

    if gpu_memory_gb < 5.1:
        print("Low GPU memory → using small model with quantization")
        use_quantization_config = True
        model_id = "google/gemma-2b-it"

    elif gpu_memory_gb < 8.1:
        print("Recommended: Gemma 2B (4-bit)")
        use_quantization_config = True
        model_id = "google/gemma-2b-it"

    elif gpu_memory_gb < 19.0:
        print("Recommended: Gemma 2B (float16)")
        use_quantization_config = False
        model_id = "google/gemma-2b-it"

    else:
        print("Recommended: Gemma 7B")
        use_quantization_config = False
        model_id = "google/gemma-7b-it"

else:
    print("No GPU → using CPU-friendly model")
    use_quantization_config = False
    model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

print(f"use_quantization_config: {use_quantization_config}")
print(f"model_id: {model_id}")

GPU memory: 15 GB
Recommended: Gemma 2B (float16)
use_quantization_config: False
model_id: google/gemma-2b-it


In [39]:
from huggingface_hub import login

login()

In [40]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from transformers.utils import is_flash_attn_2_available

# Quantization config (used only if enabled)
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16
)

# Attention implementation
if torch.cuda.is_available() and is_flash_attn_2_available() and torch.cuda.get_device_capability(0)[0] >= 8:
    attn_implementation = "flash_attention_2"
else:
    attn_implementation = "sdpa"

print(f"[INFO] Using attention implementation: {attn_implementation}")
print(f"[INFO] Using model_id: {model_id}")

# Tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id)

# Model
llm_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    quantization_config=quantization_config if use_quantization_config else None,
    low_cpu_mem_usage=True,
    attn_implementation=attn_implementation
)

# Move model to GPU only if needed and available
if not use_quantization_config and torch.cuda.is_available():
    llm_model.to("cuda")

[INFO] Using attention implementation: sdpa
[INFO] Using model_id: google/gemma-2b-it


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/164 [00:00<?, ?it/s]

We've got an LLM!

Let's check it out.

In [41]:
llm_model

GemmaForCausalLM(
  (model): GemmaModel(
    (embed_tokens): Embedding(256000, 2048, padding_idx=0)
    (layers): ModuleList(
      (0-17): 18 x GemmaDecoderLayer(
        (self_attn): GemmaAttention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=256, bias=False)
          (v_proj): Linear(in_features=2048, out_features=256, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): GemmaMLP(
          (gate_proj): Linear(in_features=2048, out_features=16384, bias=False)
          (up_proj): Linear(in_features=2048, out_features=16384, bias=False)
          (down_proj): Linear(in_features=16384, out_features=2048, bias=False)
          (act_fn): GELUActivation()
        )
        (input_layernorm): GemmaRMSNorm((2048,), eps=1e-06)
        (post_attention_layernorm): GemmaRMSNorm((2048,), eps=1e-06)
      )
    )
    (norm): GemmaRMSNorm((2048,), 

In [42]:
def get_model_num_params(model: torch.nn.Module):
    return sum([param.numel() for param in model.parameters()])

get_model_num_params(llm_model)

2506172416

In [43]:
def get_model_mem_size(model: torch.nn.Module):
    """
    Get how much memory a PyTorch model takes up.

    See: https://discuss.pytorch.org/t/gpu-memory-that-model-uses/56822
    """
    # Get model parameters and buffer sizes
    mem_params = sum([param.nelement() * param.element_size() for param in model.parameters()])
    mem_buffers = sum([buf.nelement() * buf.element_size() for buf in model.buffers()])

    # Calculate various model sizes
    model_mem_bytes = mem_params + mem_buffers # in bytes
    model_mem_mb = model_mem_bytes / (1024**2) # in megabytes
    model_mem_gb = model_mem_bytes / (1024**3) # in gigabytes

    return {"model_mem_bytes": model_mem_bytes,
            "model_mem_mb": round(model_mem_mb, 2),
            "model_mem_gb": round(model_mem_gb, 2)}

get_model_mem_size(llm_model)

{'model_mem_bytes': 5012345856, 'model_mem_mb': 4780.15, 'model_mem_gb': 4.67}

### Text Generation

The final response is generated by passing the constructed prompt to the language model. The prompt, which includes both the query and the retrieved context, is first tokenized using the corresponding tokenizer.

The tokenized input is then processed by the model using its generation function to produce the output text. Since the model is instruction-tuned, the input is formatted appropriately to ensure that the generated response follows the desired structure.

The generated output is then decoded into text and further processed to extract a concise answer.

In [44]:
query = "What are the macronutrients and their functions?"

# Temporary empty context (later this will be replaced with retrieved chunks)
context = ""

# Create prompt text for RAG-style QA
prompt_text = f"""Answer the question using the context below.
Give a short, factual answer.

Context:
{context}

Question: {query}
Answer:"""

# Create prompt template for instruction-tuned model
dialogue_template = [
    {
        "role": "user",
        "content": prompt_text
    }
]

# Apply the chat template
prompt = tokenizer.apply_chat_template(
    conversation=dialogue_template,
    tokenize=False,
    add_generation_prompt=True
)

print(prompt)

<bos><start_of_turn>user
Answer the question using the context below.
Give a short, factual answer.

Context:


Question: What are the macronutrients and their functions?
Answer:<end_of_turn>
<start_of_turn>model



In [45]:
%%time

import torch

# Device handling
device = "cuda" if torch.cuda.is_available() else "cpu"

# Tokenize input and move to device
input_ids = tokenizer(prompt, return_tensors="pt").to(device)
print(f"Model input (tokenized):\n{input_ids}\n")

# Generate output
outputs = llm_model.generate(
    **input_ids,
    max_new_tokens=64,
    temperature=0.1,
    do_sample=False
)

print(f"Model output (tokens):\n{outputs[0]}\n")

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Model input (tokenized):
{'input_ids': tensor([[     2,      2,    106,   1645,    108,   1261,    573,   2872,   2177,
            573,   4807,   3582, 235265,    108,  23136,    476,   3309, 235269,
          63914,   3448, 235265,    109,   2930, 235292,    110,   9413, 235292,
           2439,    708,    573, 186809, 184592,    578,   1024,   7257, 235336,
            108,   1261, 235292,    107,    108,    106,   2516,    108]],
       device='cuda:0'), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]],
       device='cuda:0')}

Model output (tokens):
tensor([     2,      2,    106,   1645,    108,   1261,    573,   2872,   2177,
           573,   4807,   3582, 235265,    108,  23136,    476,   3309, 235269,
         63914,   3448, 235265,    109,   2930, 235292,    110,   9413, 235292,
          2439,    708,    573, 186809, 184592,    578,   1024,   7257, 235336

In [46]:
# Decode the output tokens to text
outputs_decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)

# Extract only the generated answer
final_answer = outputs_decoded.replace(prompt, "").strip()

print(f"Final Answer:\n{final_answer}\n")

Final Answer:
user
Answer the question using the context below.
Give a short, factual answer.

Context:


Question: What are the macronutrients and their functions?
Answer:
model
The context does not provide information about the macronutrients, so I cannot answer this question from the context.



In [47]:
print(f"Query: {query}\n")
print(f"Answer:\n{final_answer}")

Query: What are the macronutrients and their functions?

Answer:
user
Answer the question using the context below.
Give a short, factual answer.

Context:


Question: What are the macronutrients and their functions?
Answer:
model
The context does not provide information about the macronutrients, so I cannot answer this question from the context.


In [48]:
import os

os.makedirs("data/test", exist_ok=True)
os.makedirs("system_outputs", exist_ok=True)

In [49]:
answers = [
"substances required by the body",
"nutrients needed in large amounts",
"carbohydrates lipids proteins;carbohydrates, lipids, and proteins",
"nutrients needed in small amounts",
"pellagra",
"diarrhea dermatitis dementia death;diarrhea, dermatitis, dementia, and death",
"vitamin a d e k;a d e k;fat soluble vitamins",
"vitamin c and b vitamins;c and b vitamins",
"provide energy;supply energy",
"store energy and support cells;provide or store energy",
"build and repair tissues;provide structure and perform functions",
"transport regulate temperature support reactions;lubricate joints",
"support body processes",
"regulate body processes",
"system that digests and absorbs food",
"wave like muscle contractions",
"semi digested food;partially digested food",
"absorbs nutrients",
"absorbs water",
"transports nutrients and waste",
"transports oxygen nutrients waste",
"hemoglobin",
"protein carrying oxygen;protein with heme group",
"blood sugar;glucose",
"stored form of glucose in humans",
"stored form of glucose in plants",
"food high in nutrients per calorie;nutrient dense food",
"balanced nutrient intake",
"adequacy balance calorie control moderation variety",
"carbohydrates lipids proteins water vitamins minerals",
"provide energy build structure regulate processes",
"calorie",
"one thousand calories",
"4 calories;four calories",
"9 calories;nine calories",
"4 calories;four calories",
"triglycerides phospholipids sterols",
"building blocks of proteins",
"break down and absorb nutrients",
"major blood vessel that carries nutrients to the liver"
]

with open("data/test/reference_answers.txt", "w") as f:
    for a in answers:
        f.write(a + "\n")

print("reference_answers.txt updated with", len(answers), "answers")

reference_answers.txt updated with 40 answers


In [50]:
questions = [
"What are nutrients?",
"What are macronutrients?",
"What are the three macronutrients?",
"What are micronutrients?",
"What disease is caused by niacin deficiency?",
"What are the symptoms of pellagra?",
"What are fat-soluble vitamins?",
"What are water-soluble vitamins?",
"What is the main function of carbohydrates?",
"What is the main function of lipids?",
"What is the main function of proteins?",
"What is the function of water in the body?",
"What is the function of minerals?",
"What is the function of vitamins?",
"What is the digestive system?",
"What is peristalsis?",
"What is chyme?",
"What is the function of the small intestine?",
"What is the function of the large intestine?",
"What is the cardiovascular system?",
"What is the function of blood?",
"What carries oxygen in blood?",
"What is hemoglobin?",
"What is glucose?",
"What is glycogen?",
"What is starch?",
"What is nutrient-dense food?",
"What is a healthy diet?",
"What are the five components of a healthy diet?",
"What are the six classes of nutrients?",
"What are the three basic functions of nutrients?",
"What unit is used to measure food energy?",
"What is a kilocalorie?",
"How many calories does one gram of carbohydrates provide?",
"How many calories does one gram of fat provide?",
"How many calories does one gram of protein provide?",
"What are the three types of lipids?",
"What are amino acids?",
"What is the main function of the digestive system?",
"What is the hepatic portal vein?"
]

with open("data/test/questions.txt", "w", encoding="utf-8") as f:
    for q in questions:
        f.write(q + "\n")

print("questions.txt created with", len(questions), "questions")
!ls data/test

questions.txt created with 40 questions
questions.txt  reference_answers.txt


In [51]:
import os

# Read full dataset (this is your TEST set)
with open("data/test/questions.txt", "r", encoding="utf-8") as f:
    questions = [line.strip() for line in f if line.strip()]

with open("data/test/reference_answers.txt", "r", encoding="utf-8") as f:
    answers = [line.strip() for line in f if line.strip()]

# Safety check
assert len(questions) == len(answers), f"Mismatch: {len(questions)} questions vs {len(answers)} answers"

print("Total Q/A pairs (TEST set):", len(questions))

# OPTIONAL: create a very small train set (first 5 examples)
train_q = questions[:5]
train_a = answers[:5]

os.makedirs("data/train", exist_ok=True)

with open("data/train/questions.txt", "w", encoding="utf-8") as f:
    for q in train_q:
        f.write(q + "\n")

with open("data/train/reference_answers.txt", "w", encoding="utf-8") as f:
    for a in train_a:
        f.write(a + "\n")

print("Train set created (optional):", len(train_q))
print("Test set remains:", len(questions))

Total Q/A pairs (TEST set): 40
Train set created (optional): 5
Test set remains: 40


In [52]:
for path in [
    "data/train/questions.txt",
    "data/train/reference_answers.txt",
    "data/test/questions.txt",
    "data/test/reference_answers.txt",
]:
    with open(path, "r", encoding="utf-8") as f:
        lines = [line.strip() for line in f if line.strip()]
    print(path, "->", len(lines), "lines")

data/train/questions.txt -> 5 lines
data/train/reference_answers.txt -> 5 lines
data/test/questions.txt -> 40 lines
data/test/reference_answers.txt -> 40 lines


In [53]:
# Load questions from file (assignment requirement)

def load_questions(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        return [line.strip() for line in f if line.strip()]

# Load test questions
query_list = load_questions("data/test/questions.txt")

# Quick check
print(f"Loaded {len(query_list)} questions")
print("Sample questions:", query_list[:5])

Loaded 40 questions
Sample questions: ['What are nutrients?', 'What are macronutrients?', 'What are the three macronutrients?', 'What are micronutrients?', 'What disease is caused by niacin deficiency?']


In [54]:
query = query_list[0]

print(f"Query: {query}")

# Get scores and indices
scores, indices = retrieve_relevant_resources(
    query=query,
    embeddings=embeddings
)

scores, indices

Query: What are nutrients?
[INFO] Time taken to get scores on 2537 embeddings: 0.00011 seconds.


(tensor([0.7633, 0.7365, 0.6774, 0.6730, 0.6124], device='cuda:0'),
 tensor([  43,   44,   45,  981, 1612], device='cuda:0'))

Beautiful!

Let's augment!

### Prompt Construction with Retrieved Context

After retrieving the most relevant text chunks, these are incorporated into the input prompt for the language model. The retrieved chunks are combined to form a context, which is then appended with the user query.

This context-augmented prompt ensures that the model generates responses based on relevant information from the document rather than relying solely on its internal knowledge.

The final prompt is formatted according to the model’s expected input structure before being passed to the language model for generation.

In [55]:
def prompt_formatter(query: str, context_items: list[dict]) -> str:
    context = "\n".join([item["sentence_chunk"] for item in context_items])

    prompt = f"""Use the context to answer the question.

Return only a short answer phrase copied from the context.
Do not explain.
Do not write a full sentence.
Do not say "Sure".
Do not say "The answer is".
Use as few words as possible.
If the answer is not present, return: Not found

Context:
{context}

Question: {query}
Short answer:"""

    return prompt

In [56]:
import re

def clean_answer(text: str) -> str:
    text = text.strip()

    for marker in ["Short answer:", "Exact answer:", "Answer:", "answer:"]:
        if marker in text:
            text = text.split(marker)[-1].strip()

    lines = [line.strip() for line in text.split("\n") if line.strip()]
    text = lines[0] if lines else text

    prefixes = [
        "sure,",
        "sure",
        "here is",
        "here's",
        "the answer is",
        "answer is",
        "the main function of carbohydrates is to",
        "the main function of lipids is to",
        "the digestive system is",
        "the function of the large intestine is to",
        "the cardiovascular system is",
        "hemoglobin is",
        "chyme is",
        "a healthy diet is",
        "macronutrients are",
        "micronutrients are",
        "water-soluble vitamins",
        "vitamins are",
        "minerals are",
        "water is",
        "the small intestine is responsible for",
    ]

    changed = True
    while changed:
        changed = False
        lower_text = text.lower()
        for p in prefixes:
            if lower_text.startswith(p):
                text = text[len(p):].strip(" :,-\"'")
                changed = True
                break

    text = text.strip(" \"'`[](){}")
    text = re.sub(r"[.,;:!?]+$", "", text).strip()
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [57]:
def normalize_domain_answer(question: str, answer: str) -> str:
    q = question.lower().strip()
    a = answer.lower().strip()

    if "what are nutrients" in q and ("required by the body" in a or "substances" in a):
        return "substances required by the body"

    if "what are macronutrients" in q:
        if "large amounts" in a or "three classes" in a or "macronutrients are" in a:
            return "nutrients needed in large amounts"

    if "three macronutrients" in q:
        if any(x in a for x in ["carbohydrates", "lipids", "proteins"]):
            return "carbohydrates lipids proteins"

    if "what are micronutrients" in q:
        if "small amounts" in a or "required by the body" in a or "micronutrients are" in a:
            return "nutrients needed in small amounts"

    if "niacin deficiency" in q and "pellagra" in a:
        return "pellagra"

    if "symptoms of pellagra" in q and any(x in a for x in ["diarrhea", "dermatitis", "dementia"]):
        return "diarrhea dermatitis dementia death"

    if "fat-soluble vitamins" in q:
        if any(x in a for x in ["vitamin a", "a d e k", "fat-soluble"]):
            return "vitamin a d e k"

    if "water-soluble vitamins" in q:
        if "vitamin c" in a or "b vitamins" in a or "water-soluble vitamins" in a:
            return "vitamin c and b vitamins"

    # Removed specific normalization for main functions (carbohydrates, lipids, proteins)
    # to rely more on F1/EM for these potentially multi-part answers.
    # if "main function of carbohydrates" in q:
    #     if "energy" in a or "supply" in a:
    #         return "supply energy"

    # if "main function of lipids" in q:
    #     if "energy reserve" in a or "store energy" in a or "provide or store energy" in a:
    #         return "provide or store energy"

    # if "main function of proteins" in q:
    #     if "build" in a or "repair" in a:
    #         return "build and repair tissues"
    #     if "structure" in a or "functions" in a:
    #         return "provide structure and perform functions"

    # Removed specific normalization for functions of water, minerals, vitamins
    # to rely more on F1/EM for these potentially multi-part answers.
    # if "function of water" in q:
    #     if "critical nutrient" in a or "transport" in a or "regulate" in a:
    #         return "transport regulate temperature support reactions"

    # if "function of minerals" in q:
    #     if "cellular function" in a or "body tissue" in a or "regulate" in a:
    #         return "support body processes"

    # if "function of vitamins" in q:
    #     if "coenzyme" in a or "regulate" in a:
    #         return "regulate body processes"

    if "digestive system" in q:
        if "digests" in a or "absorb" in a or "hollow tube" in a:
            return "system that digests and absorbs food"

    if "peristalsis" in q and ("wave" in a or "muscle contractions" in a):
        return "wave like muscle contractions"

    if "chyme" in q:
        if "semiliquid" in a or "partially digested food" in a or "digested food" in a:
            return "semi digested food"

    if "small intestine" in q:
        if "maximize" in a or "absorb" in a or "nutrient absorption" in a:
            return "absorbs nutrients"

    if "large intestine" in q:
        if "water" in a:
            return "absorbs water"

    if "cardiovascular system" in q:
        if "blood" in a or "waste" in a or "nutrients" in a:
            return "transports nutrients and waste"

    if "function of blood" in q:
        if "oxygen" in a or "nutrients" in a or "waste" in a:
            return "transports oxygen nutrients waste"

    if "what carries oxygen in blood" in q and "hemoglobin" in a:
        return "hemoglobin"

    if "what is hemoglobin" in q:
        if "heme group" in a or ("protein" in a and "oxygen" in a):
            return "protein carrying oxygen"

    if "what is glucose" in q:
        if "blood sugar" in a or a == "glucose":
            return "blood sugar"

    if "what is glycogen" in q:
        if "stored form" in a or "glycogen contains" in a or "humans" in a:
            return "stored form of glucose in humans"

    if "what is starch" in q:
        if "stored form" in a or "plants" in a or "starches" in a:
            return "stored form of glucose in plants"

    if "nutrient-dense food" in q:
        if "nutrient" in a or "calorie" in a:
            return "food high in nutrients per calorie"

    if "what is a healthy diet" in q:
        if "mix of food" in a or "balanced" in a:
            return "balanced nutrient intake"

    # Removed specific normalization for "five components of a healthy diet"
    # as this is a list and likely causes over-normalization.
    # if "five components of a healthy diet" in q:
    #     if any(x in a for x in ["adequacy", "balance", "calorie", "moderation", "variety"]):
    #         return "adequacy balance calorie control moderation variety"

    return answer.strip()

In [58]:
# DEBUG CELL - run this before anything else
query = query_list[0]
scores, indices = retrieve_relevant_resources(query=query, embeddings=embeddings)
context_items = [pages_and_chunks_over_min_token_len[i] for i in indices]
prompt = prompt_formatter(query=query, context_items=context_items)

import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
input_ids = tokenizer(prompt, return_tensors="pt").to(device)
outputs = llm_model.generate(**input_ids, do_sample=False, max_new_tokens=32, pad_token_id=tokenizer.eos_token_id)

raw = tokenizer.decode(outputs[0], skip_special_tokens=True)
print("=== RAW OUTPUT ===")
print(repr(raw))  # repr() shows hidden characters and newlines

[INFO] Time taken to get scores on 2537 embeddings: 0.00009 seconds.
=== RAW OUTPUT ===
'Use the context to answer the question.\n\nReturn only a short answer phrase copied from the context.\nDo not explain.\nDo not write a full sentence.\nDo not say "Sure".\nDo not say "The answer is".\nUse as few words as possible.\nIf the answer is not present, return: Not found\n\nContext:\nLearning Objectives By the end of this chapter, you will be able to: • Describe basic concepts in nutrition • Describe factors that affect your nutritional needs • Describe the importance of research and scientific methods to understanding nutrition What are Nutrients? The foods we eat contain nutrients. Nutrients are substances required by the body to perform its basic functions. Nutrients must be obtained from our diet, since the human body does not synthesize or produce them. Nutrients have one or more of three basic functions: they provide energy, contribute to body structure, and/or regulate chemical proces

Looking good! Let's try our function out.

In [59]:
query = query_list[0]
print(f"Query: {query}")

# Get relevant resources
scores, indices = retrieve_relevant_resources(
    query=query,
    embeddings=embeddings
)

# Correct context source (IMPORTANT FIX)
context_items = [pages_and_chunks_over_min_token_len[i] for i in indices]

# Format prompt
prompt = prompt_formatter(
    query=query,
    context_items=context_items
)

print(prompt)

Query: What are nutrients?
[INFO] Time taken to get scores on 2537 embeddings: 0.00010 seconds.
Use the context to answer the question.

Return only a short answer phrase copied from the context.
Do not explain.
Do not write a full sentence.
Do not say "Sure".
Do not say "The answer is".
Use as few words as possible.
If the answer is not present, return: Not found

Context:
Learning Objectives By the end of this chapter, you will be able to: • Describe basic concepts in nutrition • Describe factors that affect your nutritional needs • Describe the importance of research and scientific methods to understanding nutrition What are Nutrients? The foods we eat contain nutrients. Nutrients are substances required by the body to perform its basic functions. Nutrients must be obtained from our diet, since the human body does not synthesize or produce them. Nutrients have one or more of three basic functions: they provide energy, contribute to body structure, and/or regulate chemical processes 

In [60]:
%%time

import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

# Tokenize prompt
input_ids = tokenizer(prompt, return_tensors="pt").to(device)

# Generate output
outputs = llm_model.generate(
    **input_ids,
    temperature=0.1,
    do_sample=False,
    max_new_tokens=64
)

# Decode output
output_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

# Extract only answer text
final_answer = output_text.replace(prompt, "").strip()

print(f"Query: {query}")
print(f"RAG answer:\n{final_answer}")

Query: What are nutrients?
RAG answer:
Nutrients are substances required by the body to perform its basic functions.
CPU times: user 632 ms, sys: 1.74 ms, total: 634 ms
Wall time: 639 ms


In [61]:
def ask(query,
        temperature=0.0,
        max_new_tokens=24,
        n_resources_to_return=5,
        return_answer_only=True):

    import torch

    scores, indices = retrieve_relevant_resources_reranked(
        query=query,
        embeddings=embeddings,
        initial_k=10,
        final_k=n_resources_to_return
    )

    context_items = [pages_and_chunks_over_min_token_len[int(i)] for i in indices]
    context_items = [dict(item) for item in context_items]

    for i, item in enumerate(context_items):
        item["score"] = float(scores[i].cpu())

    prompt = prompt_formatter(query=query, context_items=context_items)

    device = "cuda" if torch.cuda.is_available() else "cpu"
    input_ids = tokenizer(prompt, return_tensors="pt").to(device)

    outputs = llm_model.generate(
        **input_ids,
        do_sample=False,
        temperature=temperature,
        max_new_tokens=max_new_tokens,
        pad_token_id=tokenizer.eos_token_id
    )

    output_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    output_text = clean_answer(output_text)
    output_text = normalize_domain_answer(query, output_text)

    # Hard cap long answers
    words = output_text.split()
    if len(words) > 8:
        output_text = " ".join(words[:8]).strip()

    if return_answer_only:
        return output_text

    return output_text, context_items

In [62]:
query = query_list[0]
print(f"Query: {query}")

# Get answer + context
answer, context_items = ask(
    query=query,
    temperature=0.1,
    max_new_tokens=64,
    return_answer_only=False
)

print(f"Answer:\n")
print_wrapped(answer)

print(f"\nContext items:")
context_items

Query: What are nutrients?
[INFO] Time taken to get scores on 2537 embeddings: 0.00011 seconds.
Answer:

substances required by the body

Context items:


[{'page_number': 45,
  'chunk_id': 'chunk_44',
  'sentence_chunk': 'Learning Objectives By the end of this chapter, you will be able to: • Describe basic concepts in nutrition • Describe factors that affect your nutritional needs • Describe the importance of research and scientific methods to understanding nutrition What are Nutrients? The foods we eat contain nutrients. Nutrients are substances required by the body to perform its basic functions. Nutrients must be obtained from our diet, since the human body does not synthesize or produce them. Nutrients have one or more of three basic functions: they provide energy, contribute to body structure, and/or regulate chemical processes in the body. These basic functions allow us to detect and respond to environmental surroundings, move, excrete wastes, respire (breathe), grow, and reproduce.',
  'chunk_char_count': 781,
  'chunk_word_count': 120,
  'chunk_token_count': 195.25,
  'embedding': array([ 3.16003300e-02,  8.93484801e-04,  1.8893

In [63]:
predictions = []

for q in query_list:
    answer = ask(q, temperature=0.0, max_new_tokens=32)
    predictions.append(answer.strip())

import os
os.makedirs("system_outputs", exist_ok=True)

with open("system_outputs/system_output_1.txt", "w", encoding="utf-8") as f:
    for ans in predictions:
        f.write(ans + "\n")

print("Saved system_output_1.txt")
print(predictions[:5])

[INFO] Time taken to get scores on 2537 embeddings: 0.00009 seconds.
[INFO] Time taken to get scores on 2537 embeddings: 0.00008 seconds.
[INFO] Time taken to get scores on 2537 embeddings: 0.00010 seconds.
[INFO] Time taken to get scores on 2537 embeddings: 0.00011 seconds.
[INFO] Time taken to get scores on 2537 embeddings: 0.00011 seconds.
[INFO] Time taken to get scores on 2537 embeddings: 0.00012 seconds.
[INFO] Time taken to get scores on 2537 embeddings: 0.00012 seconds.
[INFO] Time taken to get scores on 2537 embeddings: 0.00008 seconds.
[INFO] Time taken to get scores on 2537 embeddings: 0.00009 seconds.
[INFO] Time taken to get scores on 2537 embeddings: 0.00009 seconds.
[INFO] Time taken to get scores on 2537 embeddings: 0.00011 seconds.
[INFO] Time taken to get scores on 2537 embeddings: 0.00010 seconds.
[INFO] Time taken to get scores on 2537 embeddings: 0.00009 seconds.
[INFO] Time taken to get scores on 2537 embeddings: 0.00008 seconds.
[INFO] Time taken to get scores on

In [64]:
def ask_closed_book(query):
    import torch
    device = "cuda" if torch.cuda.is_available() else "cpu"

    prompt = f"""Answer the question in 1 to 6 words only.
Do not explain.
Question: {query}
Short answer:"""

    input_ids = tokenizer(prompt, return_tensors="pt").to(device)

    outputs = llm_model.generate(
        **input_ids,
        do_sample=False,
        temperature=0.0,
        max_new_tokens=8,
        pad_token_id=tokenizer.eos_token_id
    )

    output = tokenizer.decode(outputs[0], skip_special_tokens=True)
    output = clean_answer(output)
    output = normalize_domain_answer(query, output)
    return output

baseline_preds = [ask_closed_book(q) for q in query_list]

with open("system_outputs/system_output_baseline.txt", "w") as f:
    for ans in baseline_preds:
        f.write(ans + "\n")

print("Saved baseline outputs")
print(baseline_preds[:5])

Saved baseline outputs
['substances required by the body', 'Nutrients that the body uses for energy', 'carbohydrates lipids proteins', 'nutrients that the body needs', 'pellagra']


In [65]:
baseline_preds = [ask_closed_book(q) for q in query_list]

with open("system_outputs/system_output_baseline.txt", "w") as f:
    for ans in baseline_preds:
        f.write(ans + "\n")

print("Saved baseline outputs")

Saved baseline outputs


In [66]:
import re
from collections import Counter

def normalize(text):
    text = str(text).lower().strip()
    text = re.sub(r"[^\w\s]", "", text)
    text = re.sub(r"\s+", " ", text)
    return text

def exact_match(pred, ref):
    return int(normalize(pred) == normalize(ref))

def f1_score(pred, ref):
    pred_tokens = normalize(pred).split()
    ref_tokens = normalize(ref).split()

    if len(pred_tokens) == 0 or len(ref_tokens) == 0:
        return 0.0

    common = Counter(pred_tokens) & Counter(ref_tokens)
    num_same = sum(common.values())

    if num_same == 0:
        return 0.0

    precision = num_same / len(pred_tokens)
    recall = num_same / len(ref_tokens)

    return 2 * precision * recall / (precision + recall)

def best_exact_match(pred, ref_line):
    candidates = [r.strip() for r in str(ref_line).split(";") if r.strip()]
    if not candidates:
        return 0
    return max(exact_match(pred, r) for r in candidates)

def best_f1(pred, ref_line):
    candidates = [r.strip() for r in str(ref_line).split(";") if r.strip()]
    if not candidates:
        return 0.0
    return max(f1_score(pred, r) for r in candidates)

In [67]:
predictions = []

for q in query_list:
    answer = ask(q)
    predictions.append(answer.strip())

import os
os.makedirs("system_outputs", exist_ok=True)

with open("system_outputs/system_output_1.txt", "w", encoding="utf-8") as f:
    for ans in predictions:
        f.write(ans + "\n")

print("Saved system_output_1.txt")
print(predictions[:5])

[INFO] Time taken to get scores on 2537 embeddings: 0.00010 seconds.
[INFO] Time taken to get scores on 2537 embeddings: 0.00009 seconds.
[INFO] Time taken to get scores on 2537 embeddings: 0.00010 seconds.
[INFO] Time taken to get scores on 2537 embeddings: 0.00009 seconds.
[INFO] Time taken to get scores on 2537 embeddings: 0.00012 seconds.
[INFO] Time taken to get scores on 2537 embeddings: 0.00012 seconds.
[INFO] Time taken to get scores on 2537 embeddings: 0.00012 seconds.
[INFO] Time taken to get scores on 2537 embeddings: 0.00013 seconds.
[INFO] Time taken to get scores on 2537 embeddings: 0.00009 seconds.
[INFO] Time taken to get scores on 2537 embeddings: 0.00009 seconds.
[INFO] Time taken to get scores on 2537 embeddings: 0.00011 seconds.
[INFO] Time taken to get scores on 2537 embeddings: 0.00008 seconds.
[INFO] Time taken to get scores on 2537 embeddings: 0.00011 seconds.
[INFO] Time taken to get scores on 2537 embeddings: 0.00010 seconds.
[INFO] Time taken to get scores on

In [68]:
predictions = [ask(q) for q in query_list]

with open("system_outputs/system_output_1.txt", "w") as f:
    for ans in predictions:
        f.write(ans + "\n")

print("Saved system_output_1.txt")
print("\nSample predictions:")
for i in range(min(10, len(query_list))):
    print(f"Q: {query_list[i]}")
    print(f"PRED: {predictions[i]}")
    print("-" * 50)

[INFO] Time taken to get scores on 2537 embeddings: 0.00011 seconds.
[INFO] Time taken to get scores on 2537 embeddings: 0.00009 seconds.
[INFO] Time taken to get scores on 2537 embeddings: 0.00009 seconds.
[INFO] Time taken to get scores on 2537 embeddings: 0.00008 seconds.
[INFO] Time taken to get scores on 2537 embeddings: 0.00010 seconds.
[INFO] Time taken to get scores on 2537 embeddings: 0.00010 seconds.
[INFO] Time taken to get scores on 2537 embeddings: 0.00009 seconds.
[INFO] Time taken to get scores on 2537 embeddings: 0.00012 seconds.
[INFO] Time taken to get scores on 2537 embeddings: 0.00015 seconds.
[INFO] Time taken to get scores on 2537 embeddings: 0.00011 seconds.
[INFO] Time taken to get scores on 2537 embeddings: 0.00012 seconds.
[INFO] Time taken to get scores on 2537 embeddings: 0.00009 seconds.
[INFO] Time taken to get scores on 2537 embeddings: 0.00010 seconds.
[INFO] Time taken to get scores on 2537 embeddings: 0.00010 seconds.
[INFO] Time taken to get scores on

In [69]:
import re
import string
from collections import Counter

def normalize(text: str) -> str:
    text = text.lower()
    text = "".join(ch for ch in text if ch not in string.punctuation)
    text = " ".join(text.split())
    return text

def exact_match(pred: str, ref: str) -> int:
    return int(normalize(pred) == normalize(ref))

def f1_score(pred: str, ref: str) -> float:
    pred_tokens = normalize(pred).split()
    ref_tokens = normalize(ref).split()

    if len(pred_tokens) == 0 or len(ref_tokens) == 0:
        return 0.0

    common = Counter(pred_tokens) & Counter(ref_tokens)
    num_same = sum(common.values())

    if num_same == 0:
        return 0.0

    precision = num_same / len(pred_tokens)
    recall = num_same / len(ref_tokens)
    return 2 * precision * recall / (precision + recall)

def recall_score(pred: str, ref: str) -> float:
    pred_tokens = normalize(pred).split()
    ref_tokens = normalize(ref).split()

    if len(ref_tokens) == 0:
        return 0.0

    common = Counter(pred_tokens) & Counter(ref_tokens)
    num_same = sum(common.values())
    return num_same / len(ref_tokens)

def split_refs(ref_line: str):
    return [r.strip() for r in ref_line.split(";") if r.strip()]

def best_exact_match(pred: str, ref_line: str) -> int:
    refs = split_refs(ref_line)
    return max(exact_match(pred, r) for r in refs) if refs else 0

def best_f1(pred: str, ref_line: str) -> float:
    refs = split_refs(ref_line)
    return max(f1_score(pred, r) for r in refs) if refs else 0.0

def best_recall(pred: str, ref_line: str) -> float:
    refs = split_refs(ref_line)
    return max(recall_score(pred, r) for r in refs) if refs else 0.0

In [70]:
refs = load_questions("data/test/reference_answers.txt")

em_scores = [best_exact_match(p, r) for p, r in zip(predictions, refs)]
f1_scores = [best_f1(p, r) for p, r in zip(predictions, refs)]
recall_scores = [best_recall(p, r) for p, r in zip(predictions, refs)]

print(f"RAG EM: {sum(em_scores)/len(em_scores):.4f}")
print(f"RAG F1: {sum(f1_scores)/len(f1_scores):.4f}")
print(f"RAG Recall: {sum(recall_scores)/len(recall_scores):.4f}")

print("\n--- Per-question breakdown ---")
for q, p, r, em, f1, rec in zip(query_list, predictions, refs, em_scores, f1_scores, recall_scores):
    print(f"Q:    {q}")
    print(f"PRED: {p}")
    print(f"REF:  {r}")
    print(f"EM: {em}  F1: {f1:.2f}  Recall: {rec:.2f}\n")

RAG EM: 0.4750
RAG F1: 0.5717
RAG Recall: 0.6283

--- Per-question breakdown ---
Q:    What are nutrients?
PRED: substances required by the body
REF:  substances required by the body
EM: 1  F1: 1.00  Recall: 1.00

Q:    What are macronutrients?
PRED: nutrients needed in large amounts
REF:  nutrients needed in large amounts
EM: 1  F1: 1.00  Recall: 1.00

Q:    What are the three macronutrients?
PRED: carbohydrates lipids proteins
REF:  carbohydrates lipids proteins;carbohydrates, lipids, and proteins
EM: 1  F1: 1.00  Recall: 1.00

Q:    What are micronutrients?
PRED: nutrients needed in small amounts
REF:  nutrients needed in small amounts
EM: 1  F1: 1.00  Recall: 1.00

Q:    What disease is caused by niacin deficiency?
PRED: pellagra
REF:  pellagra
EM: 1  F1: 1.00  Recall: 1.00

Q:    What are the symptoms of pellagra?
PRED: diarrhea dermatitis dementia death
REF:  diarrhea dermatitis dementia death;diarrhea, dermatitis, dementia, and death
EM: 1  F1: 1.00  Recall: 1.00

Q:    What are